# LLM Web Search — Provider-Native Grounding

Let the model fetch live web results on its own — no scraping, no Serp API. Each provider exposes a **server-side search tool** that Anthropic/OpenAI run internally before generating the answer.

> **Providers**: OpenAI and Anthropic only (Gemini and Ollama excluded by design).

In [ ]:
import os

SEARCH_PROMPT = 'What are the latest developments in large language model research from the past week?'

---
## OpenAI

Uses the **Responses API** (`client.responses.create`) with the `web_search_preview` built-in tool. The model searches the web, then synthesizes an answer — all in one call.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.responses.create(
    model=OPENAI_MODEL,
    tools=[{'type': 'web_search_preview'}],
    input=SEARCH_PROMPT
)

for item in openai_resp.output:
    if item.type == 'message':
        for block in item.content:
            if block.type == 'output_text':
                print(block.text)

---
## Anthropic

Declare `web_search_20260209` as a tool; Anthropic's infrastructure executes the search and returns grounded results. The model cites sources in its reply.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=1024,
    tools=[{'type': 'web_search_20260209', 'name': 'web_search'}],
    messages=[{'role': 'user', 'content': SEARCH_PROMPT}]
)

for block in anthropic_resp.content:
    if block.type == 'text':
        print(block.text)